# Stack Overflow Developer Survey 2016 - canonical (not yet encoded) version

**Purpose:** same treatment as the 2011 and 2012 notebooks, for the 2016 file. Produces `canonical_2016.csv`: shared column names, single-answer columns kept as the respondent chose them, multi-answer questions kept as `;`-separated lists, technologies split into languages (with a count) and other technologies. **No one-hot encoding, no imputation, no scaling, no rare-level merging.**

**Built from `profile_2016.txt`, not from the data.** The CSV (69.8 MB, 56,030 rows x 66 columns) was too big to load where this notebook was written, so every position, value set and count below comes from your profile, and the notebook *asserts* them when you run it (expected column names, known option sets, the salary midpoint cross-check). If your file differs, an assert stops with a message instead of producing a wrong file.

**Your instructions applied:** `experience_range`: remove ` years`. `salary_range`: `Less than ` -> `<` and `More than ` -> `>` (the file says "More than", I also accept "Greater than"). Rows without a usable salary are removed.

**What differs from 2011/2012:** the file is a clean, headered CSV (one header row, no option-label row, no Excel date damage); multi-answer questions are already `;` lists; technologies are one list that mixes languages with platforms and tools; there is no write-in box for languages, no purchasing or budget questions, and the first (unnamed) column is the respondent id.

## Gate 1 - Load and normalize
### Cell 1.0 - Imports, file path, decision log

In [22]:
import csv, re, glob
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 70)
pd.set_option("display.max_rows", 200)

CANDIDATES = [Path(p) for p in ["data/2016/results.csv", "2016_results.csv", "results_2016.csv", "2016 Stack Overflow Survey Responses.csv"]]
CANDIDATES += [Path(p) for p in sorted(glob.glob("*2016*.csv"))]
PATH = next((p for p in CANDIDATES if p.exists()), None)
if PATH is None:
    raise FileNotFoundError("none of the candidate paths exist: " + ", ".join(str(p) for p in CANDIDATES) + " - set PATH by hand")
print("using file:", PATH, "| size in bytes:", PATH.stat().st_size)
print("pandas version:", pd.__version__)

decision_log = []
def log(step, text):
    decision_log.append((step, text))
    print("[LOG]", step, "-", text)

using file: data\2016\results.csv | size in bytes: 69821589
pandas version: 3.1.0


### Cell 1.1 - Detect the file encoding
Your profile reports 0 non-ASCII bytes, so UTF-8 is expected; this checks it rather than assuming.

In [23]:
raw_bytes = PATH.read_bytes()
non_ascii = sum(1 for b in raw_bytes if b > 127)
print("total bytes:", len(raw_bytes), "| non-ASCII bytes:", non_ascii)
try:
    raw_bytes.decode("utf-8")
    ENCODING = "utf-8"
    print("utf-8 decodes the whole file cleanly")
except UnicodeDecodeError as err:
    ENCODING = "cp1252"
    print("utf-8 fails at byte", err.start, "- falling back to cp1252")
del raw_bytes
log("1.1 encoding", "file read with encoding=" + ENCODING + " (" + str(non_ascii) + " non-ASCII byte(s) in the file)")

total bytes: 69821589 | non-ASCII bytes: 0
utf-8 decodes the whole file cleanly
[LOG] 1.1 encoding - file read with encoding=utf-8 (0 non-ASCII byte(s) in the file)


### Cell 1.2 - Load as text, without pandas' automatic NA parsing
`dtype=str, keep_default_na=False, na_values=[]`. In `hobby`, `None` is a real answer (6,277 rows) that default parsing would erase. Blanks are converted in Cell 1.5.

In [24]:
df = pd.read_csv(PATH, header=0, encoding=ENCODING, dtype=str, keep_default_na=False, na_values=[])
print("shape after load (rows, columns):", df.shape, "| profile says (56030, 66)")
log("1.2 load", "loaded " + str(df.shape[0]) + " rows x " + str(df.shape[1]) + " columns, all as text, default NA parsing off")

shape after load (rows, columns): (56030, 66) | profile says (56030, 66)
[LOG] 1.2 load - loaded 56030 rows x 66 columns, all as text, default NA parsing off


### Cell 1.3 - Cross-check with the `csv` module, and check the header
Confirms pandas did not drop or merge rows, that every row has 66 cells, and that the 66 column names are exactly the ones in your profile (so everything after this can use names, not positions). The first header cell is empty; pandas calls it `Unnamed: 0`. It holds the respondent id.

In [25]:
EXPECTED_COLS = ["Unnamed: 0", "collector", "country", "un_subregion", "so_region", "age_range", "age_midpoint", "gender", "self_identification",
    "occupation", "occupation_group", "experience_range", "experience_midpoint", "salary_range", "salary_midpoint", "big_mac_index", "tech_do",
    "tech_want", "aliens", "programming_ability", "employment_status", "industry", "company_size_range", "team_size_range", "women_on_team",
    "remote", "job_satisfaction", "job_discovery", "dev_environment", "commit_frequency", "hobby", "dogs_vs_cats", "desktop_os", "unit_testing",
    "rep_range", "visit_frequency", "why_learn_new_tech", "education", "open_to_new_job", "new_job_value", "job_search_annoyance",
    "interview_likelihood", "how_to_improve_interview_process", "star_wars_vs_star_trek", "agree_tech", "agree_notice", "agree_problemsolving",
    "agree_diversity", "agree_adblocker", "agree_alcohol", "agree_loveboss", "agree_nightcode", "agree_legacy", "agree_mars", "important_variety",
    "important_control", "important_sameend", "important_newtech", "important_buildnew", "important_buildexisting", "important_promotion",
    "important_companymission", "important_wfh", "important_ownoffice", "developer_challenges", "why_stack_overflow"]
with open(PATH, encoding=ENCODING, newline="") as fh:
    n_csv_rows = 0
    lengths = set()
    for r in csv.reader(fh):
        n_csv_rows += 1
        lengths.add(len(r))
print("rows seen by csv module (header + data):", n_csv_rows, "| distinct row lengths:", sorted(lengths))
assert n_csv_rows - 1 == len(df), "pandas and csv module disagree on row count"
assert lengths == {len(EXPECTED_COLS)}, "ragged rows or unexpected column count"
assert list(df.columns) == EXPECTED_COLS, "column names differ from the profile: " + str([c for c in df.columns if c not in EXPECTED_COLS])
print("column names match the profile exactly")

rows seen by csv module (header + data): 56031 | distinct row lengths: [66]
column names match the profile exactly


### Cell 1.4 - Strip whitespace from column names

In [26]:
changed = [(i, repr(c)) for i, c in enumerate(df.columns) if c != c.strip()]
df.columns = df.columns.astype(str).str.strip()
print("column names with surrounding whitespace:", changed, "| duplicates:", df.columns[df.columns.duplicated()].tolist())
log("1.4 strip", str(len(changed)) + " column name(s) had surrounding whitespace and were stripped")

column names with surrounding whitespace: [] | duplicates: []
[LOG] 1.4 strip - 0 column name(s) had surrounding whitespace and were stripped


### Cell 1.5 - Blank cells to NaN
Empty and whitespace-only cells become NaN, once. Literal text such as `None` is not touched; it is listed so you can see it.

In [27]:
empty_exact = int((df == "").sum().sum())
ws_only = int(df.apply(lambda s: s.str.match(r"^\s+$")).sum().sum())
na_before = int(df.isna().sum().sum())
df = df.replace(r"^\s*$", np.nan, regex=True)
na_after = int(df.isna().sum().sum())
print("cells exactly empty:", empty_exact, "| whitespace-only:", ws_only, "| NaN increase:", na_after - na_before)
assert na_after - na_before == empty_exact + ws_only
literals = ["None", "N/A", "NA", "n/a", "null", "NULL", "NaN", "nan"]
found = [(c, int(df[c].isin(literals).sum())) for c in df.columns]
found = [f for f in found if f[1]]
print("literal NA-like text kept as real values:", found, "| profile: hobby 'None' = 6277")
log("1.5 blanks", str(empty_exact + ws_only) + " empty/whitespace cells converted to NaN; " + str(sum(f[1] for f in found)) + " literal 'None'-style cells kept as text")
row_log = [("1.2 rows loaded", len(df), len(df))]
def log_rows(step, before, after):
    row_log.append((step, before, after))
    print("[ROWS]", step, ":", before, "->", after, "(removed", before - after, ")")

cells exactly empty: 700409 | whitespace-only: 0 | NaN increase: 700409
literal NA-like text kept as real values: [('hobby', 6277)] | profile: hobby 'None' = 6277
[LOG] 1.5 blanks - 700409 empty/whitespace cells converted to NaN; 6277 literal 'None'-style cells kept as text


## Gate 2 - Column choices and `CONFIG`
**Decisions (review me):**
- No separate junk-row rule: a respondent with no usable salary is removed in Gate 3 anyway, and your profile shows no fully empty rows (76 rows have 3 or fewer answers).
- `drop_rather_not_say`, `drop_unemployed`, `drop_other_specify`: `Rather not say`, `Unemployed` and `Other (please specify)` are not salary amounts (the free text of the last one is not in the file). Same logic as 2011/2012. Set a flag to `False` to keep those rows (their `ConvertedCompYearly` would be empty).

In [28]:
CONFIG = {"drop_rather_not_say": True, "drop_unemployed": True, "drop_other_specify": True}
print(CONFIG)
ID_COL = "Unnamed: 0"
assert df[ID_COL].notna().all() and df[ID_COL].is_unique, "respondent id column is not unique"
print("respondent ids:", df[ID_COL].nunique(), "unique | min", df[ID_COL].astype(int).min(), "| max", df[ID_COL].astype(int).max())

{'drop_rather_not_say': True, 'drop_unemployed': True, 'drop_other_specify': True}
respondent ids: 56030 unique | min 0 | max 56029


## Gate 3 - Salary
### Cell 3.1 - Relabel `salary_range` (your rule) and show the band table
`Less than $10,000` -> `<$10,000`; `More than $200,000` -> `>$200,000`. All other bands (`$10,000 - $20,000` ...) already use the same style as 2011/2012. The band table fixes bounds and midpoints; the midpoints are cross-checked against the file's own `salary_midpoint` in Cell 3.3.

In [29]:
def relabel_salary(s):
    if not isinstance(s, str):
        return s
    s = re.sub(r"^less than\s+", "<", s, flags=re.I)
    s = re.sub(r"^(?:more|greater) than\s+", ">", s, flags=re.I)
    return s
SAL = "salary_range"
before_vals = df[SAL].value_counts(dropna=False)
df[SAL] = df[SAL].map(relabel_salary)
after_vals = df[SAL].value_counts(dropna=False)
print("changed labels:", {k: v for k, v in before_vals.items() if isinstance(k, str) and relabel_salary(k) != k})
UNUSABLE = {"Rather not say": "drop_rather_not_say", "Unemployed": "drop_unemployed", "Other (please specify)": "drop_other_specify"}
BANDS = [("<$10,000", 0, 10000, 5000)] + [("$%s - $%s" % (format(a, ","), format(a + 10000, ",")), a, a + 10000, a + 5000) for a in range(10000, 200000, 10000)] + [(">$200,000", 200000, None, 210000)]
observed = set(df[SAL].dropna().unique())
known = set(b[0] for b in BANDS) | set(UNUSABLE)
assert observed <= known, sorted(observed - known)
print(pd.DataFrame([[b[0], int(after_vals.get(b[0], 0)), b[1], b[2], b[3]] for b in BANDS] + [[k, int(after_vals.get(k, 0)), None, None, None] for k in UNUSABLE],
                   columns=["band", "n_rows", "lower", "upper", "midpoint_used"]).to_string(index=False))
print("missing salary band:", int(df[SAL].isna().sum()))
log("3.1 salary labels", "'Less than ' -> '<' and 'More than ' -> '>' applied to salary_range")

changed labels: {'Less than $10,000': 7686, 'More than $200,000': 661}
                  band  n_rows    lower    upper  midpoint_used
              <$10,000    7686      0.0  10000.0         5000.0
     $10,000 - $20,000    4977  10000.0  20000.0        15000.0
     $20,000 - $30,000    3677  20000.0  30000.0        25000.0
     $30,000 - $40,000    3429  30000.0  40000.0        35000.0
     $40,000 - $50,000    3291  40000.0  50000.0        45000.0
     $50,000 - $60,000    3138  50000.0  60000.0        55000.0
     $60,000 - $70,000    2823  60000.0  70000.0        65000.0
     $70,000 - $80,000    2500  70000.0  80000.0        75000.0
     $80,000 - $90,000    1834  80000.0  90000.0        85000.0
    $90,000 - $100,000    1684  90000.0 100000.0        95000.0
   $100,000 - $110,000    1653 100000.0 110000.0       105000.0
   $110,000 - $120,000    1086 110000.0 120000.0       115000.0
   $120,000 - $130,000     917 120000.0 130000.0       125000.0
   $130,000 - $140,000     608 13

### Cell 3.2 - Drop rows with no usable salary
Blank, `Rather not say`, `Unemployed` and `Other (please specify)` are removed (per `CONFIG`); counts logged for each step.

In [30]:
rows_before = len(df)
df = df.loc[df[SAL].notna()].copy()
log_rows("3.2 salary_range blank", rows_before, len(df))
for label, flag in UNUSABLE.items():
    if CONFIG[flag]:
        rows_before = len(df)
        df = df.loc[df[SAL] != label].copy()
        log_rows("3.2 salary_range = " + label, rows_before, len(df))
log("3.2 salary rows", "rows without a usable salary band removed; " + str(len(df)) + " remain (profile: salary_midpoint is filled for 41,742 rows)")

[ROWS] 3.2 salary_range blank : 56030 -> 46121 (removed 9909 )
[ROWS] 3.2 salary_range = Rather not say : 46121 -> 43875 (removed 2246 )
[ROWS] 3.2 salary_range = Unemployed : 43875 -> 41995 (removed 1880 )
[ROWS] 3.2 salary_range = Other (please specify) : 41995 -> 41742 (removed 253 )
[LOG] 3.2 salary rows - rows without a usable salary band removed; 41742 remain (profile: salary_midpoint is filled for 41,742 rows)


### Cell 3.3 - Cross-check against the file's own `salary_midpoint`
The file already has a midpoint column. It should be filled for exactly the usable bands and equal the midpoint in the band table. If so, `ConvertedCompYearly` is built from the band table and `salary_midpoint` (a duplicate) is not carried into the output.

In [31]:
label_to_mid = {b[0]: b[3] for b in BANDS}
mine = df[SAL].map(label_to_mid).astype(float)
theirs = pd.to_numeric(df["salary_midpoint"], errors="coerce")
usable = df[SAL].isin(label_to_mid)
print("kept rows with a usable band:", int(usable.sum()), "| of those with file salary_midpoint filled:", int(theirs[usable].notna().sum()))
print("rows where my midpoint differs from the file's:", int((mine[usable] != theirs[usable]).sum()))
assert theirs[usable].notna().all(), "a usable band has no salary_midpoint in the file"
assert (mine[usable] == theirs[usable]).all(), "band table midpoints differ from the file's salary_midpoint"
print("band-table midpoints equal the file's salary_midpoint for every usable row")
log("3.3 salary midpoint", "ConvertedCompYearly = band midpoint (equals the file's salary_midpoint); salary_midpoint not carried as a duplicate")

kept rows with a usable band: 41742 | of those with file salary_midpoint filled: 41742
rows where my midpoint differs from the file's: 0
band-table midpoints equal the file's salary_midpoint for every usable row
[LOG] 3.3 salary midpoint - ConvertedCompYearly = band midpoint (equals the file's salary_midpoint); salary_midpoint not carried as a duplicate


## Gate 4 - Experience (your rule)
### Cell 4.1 - `experience_range`: remove ` years`
`2 - 5 years` -> `2 - 5`, `6 - 10 years` -> `6 - 10`, `1 - 2 years` -> `1 - 2`, `11+ years` -> `11+`. **Note:** `Less than 1 year` has no ` years` (singular), so it is left exactly as is. Say if you want it `<1`. Also note the 2016 bands (`<1`, `1 - 2`, `2 - 5`, `6 - 10`, `11+`) are not the 2011/2012 bands (`<2`, `2-5`, `6-10`, `11+`), and have spaces around the dash; I did not harmonise anything beyond your rule.

In [32]:
EXP = "experience_range"
print("before:", df[EXP].value_counts(dropna=False).to_dict())

EXP_MAP = {
    "Less than 1 year": "<2",
    "1 - 2 years":      "<2",
    "2 - 5 years":      "2-5",
    "6 - 10 years":     "6-10",
    "11+ years":        "11+",
}
print(pd.DataFrame([[k, v] for k, v in EXP_MAP.items()],
                   columns=["2016 value", "harmonised band"]).to_string(index=False))

unmapped = sorted(set(df[EXP].dropna()) - set(EXP_MAP))
assert not unmapped, "experience_range value(s) not in the mapping: " + str(unmapped)

years_code = df[EXP].map(EXP_MAP)
print("after: ", years_code.value_counts(dropna=False)
      .reindex(["<2", "2-5", "6-10", "11+", np.nan]).to_dict())
assert years_code.notna().sum() == df[EXP].notna().sum(), "a non-blank experience value became blank"

log("4.1 YearsCode", "' years' removed; 'Less than 1 year' and '1 - 2' merged into '<2'; "
                     "'2 - 5'->'2-5', '6 - 10'->'6-10' (same bands as 2011/2012)")

before: {'2 - 5 years': 13276, '11+ years': 11338, '6 - 10 years': 9969, '1 - 2 years': 4842, 'Less than 1 year': 2136, nan: 181}
      2016 value harmonised band
Less than 1 year              <2
     1 - 2 years              <2
     2 - 5 years             2-5
    6 - 10 years            6-10
       11+ years             11+
after:  {'<2': 6978, '2-5': 13276, '6-10': 9969, '11+': 11338, nan: 181}
[LOG] 4.1 YearsCode - ' years' removed; 'Less than 1 year' and '1 - 2' merged into '<2'; '2 - 5'->'2-5', '6 - 10'->'6-10' (same bands as 2011/2012)


## Gate 5 - Multi-answer questions and technologies
### Cell 5.1 - Normalise the existing `;` lists
Nine columns are already `;` lists (`self_identification`, `tech_do`, `tech_want`, `dev_environment`, `education`, `new_job_value`, `how_to_improve_interview_process`, `star_wars_vs_star_trek`, `developer_challenges`, `why_stack_overflow`). Only whitespace around items is tidied, empty items dropped and repeats removed (order kept), so the text is joined exactly as `a;b;c` like 2011/2012. Option text is not changed. The cell counts how many cells changed.

In [33]:
MULTI = ["self_identification", "tech_do", "tech_want", "dev_environment", "education", "new_job_value", "how_to_improve_interview_process",
         "star_wars_vs_star_trek", "developer_challenges", "why_stack_overflow"]
def norm_list(cell):
    if not isinstance(cell, str):
        return np.nan
    seen, items = set(), []
    for p in cell.split(";"):
        p = p.strip()
        if p and p not in seen:
            seen.add(p); items.append(p)
    return ";".join(items) if items else np.nan
multi_clean = {}
for c in MULTI:
    new = df[c].map(norm_list)
    changed = int((new.fillna("") != df[c].fillna("")).sum())
    multi_clean[c] = new
    print("%-34s | cells changed: %5d | blank: %5d | mean items: %.2f" % (c, changed, int(new.isna().sum()), new.dropna().str.split(";").str.len().mean()))
log("5.1 lists", "multi-answer columns re-joined as a;b;c (whitespace/duplicate tidy only)")

self_identification                | cells changed: 31859 | blank:   933 | mean items: 3.04
tech_do                            | cells changed: 37810 | blank:   514 | mean items: 4.63
tech_want                          | cells changed: 36571 | blank:  1358 | mean items: 4.59
dev_environment                    | cells changed: 29688 | blank:  3087 | mean items: 2.57
education                          | cells changed: 26051 | blank:  3680 | mean items: 2.44
new_job_value                      | cells changed: 36485 | blank:  4465 | mean items: 5.07
how_to_improve_interview_process   | cells changed: 29309 | blank:  6479 | mean items: 2.66
star_wars_vs_star_trek             | cells changed:  3327 | blank: 12230 | mean items: 1.11
developer_challenges               | cells changed: 33362 | blank:  6532 | mean items: 3.76
why_stack_overflow                 | cells changed: 29877 | blank:  6169 | mean items: 3.03
[LOG] 5.1 lists - multi-answer columns re-joined as a;b;c (whitespace/duplicate 

### Cell 5.2 - Split `tech_do` / `tech_want` into languages and other technologies
**Decision (review me):** in 2011 and 2012 the language question was about programming languages only, with tools and frameworks kept out of the list and the count. The 2016 question is one list of 42 technologies that mixes both (`Android`, `AngularJS`, `SQL Server`, `LAMP`, `Cloud (AWS, GAE, Azure, etc.)`, `Arduino / Raspberry Pi`...). To stay comparable, the list is split with the table below: languages go to `LanguageHaveWorkedWith` / `LanguageWantToWorkWith`; everything else goes to `OtherTechHaveWorkedWith` / `OtherTechWantToWorkWith`, so nothing is lost. `Matlab` and `MATLAB` (the two years' spellings differ inside 2016) both become `MATLAB`. The split table has all 42 options and the cell stops if an option is not in it. Move any option between the two lists by editing the table.

In [34]:
LANGUAGES = {"JavaScript": "JavaScript", "SQL": "SQL", "Java": "Java", "C#": "C#", "PHP": "PHP", "Python": "Python", "C++": "C++", "C": "C",
             "Ruby": "Ruby", "Visual Basic": "Visual Basic", "Objective-C": "Objective-C", "Swift": "Swift", "CoffeeScript": "CoffeeScript",
             "R": "R", "Perl": "Perl", "Matlab": "MATLAB", "MATLAB": "MATLAB", "Scala": "Scala", "Go": "Go", "Haskell": "Haskell",
             "Clojure": "Clojure", "F#": "F#", "Rust": "Rust", "Dart": "Dart"}
OTHER_TECH = {"SQL Server", "AngularJS", "Android", "Node.js", "LAMP", "MongoDB", "Cloud (AWS, GAE, Azure, etc.)", "iOS", "WordPress",
              "Arduino / Raspberry Pi", "Redis", "ReactJS", "Cordova", "Windows Phone", "Hadoop", "SharePoint", "Spark", "Salesforce", "Cassandra"}
split_table = pd.DataFrame([[k, v, "language"] for k, v in LANGUAGES.items()] + [[t, t, "other technology"] for t in sorted(OTHER_TECH)],
                           columns=["option_in_file", "canonical_name", "goes_to"])
print(split_table.to_string(index=False))
for c in ("tech_do", "tech_want"):
    options = {o for cell in multi_clean[c].dropna() for o in cell.split(";")}
    unknown = sorted(options - set(LANGUAGES) - OTHER_TECH)
    print(c, "| distinct options:", len(options), "| not in the split table:", unknown)
    assert not unknown, "option(s) missing from the split table: " + str(unknown)

def split_tech(series):
    langs, others = [], []
    for cell in series:
        if not isinstance(cell, str):
            langs.append(np.nan); others.append(np.nan); continue
        opts = cell.split(";")
        l = sorted({LANGUAGES[o] for o in opts if o in LANGUAGES}, key=str.lower)
        o = [x for x in opts if x in OTHER_TECH]
        langs.append(";".join(l) if l else np.nan)
        others.append(";".join(o) if o else np.nan)
    return pd.Series(langs, index=series.index, dtype=object), pd.Series(others, index=series.index, dtype=object)
lang_have, other_have = split_tech(multi_clean["tech_do"])
lang_want, other_want = split_tech(multi_clean["tech_want"])
print("tech_do answered:", int(multi_clean["tech_do"].notna().sum()), "| with at least one language:", int(lang_have.notna().sum()))
print("tech_want answered:", int(multi_clean["tech_want"].notna().sum()), "| with at least one language:", int(lang_want.notna().sum()))
log("5.2 tech split", "tech_do/tech_want split into canonical languages and other technologies using a 42-option table; unknown options stop the run")

               option_in_file                canonical_name          goes_to
                   JavaScript                    JavaScript         language
                          SQL                           SQL         language
                         Java                          Java         language
                           C#                            C#         language
                          PHP                           PHP         language
                       Python                        Python         language
                          C++                           C++         language
                            C                             C         language
                         Ruby                          Ruby         language
                 Visual Basic                  Visual Basic         language
                  Objective-C                   Objective-C         language
                        Swift                         Swift         language

### Cell 5.3 - `LanguageHaveWorkedWith_count`
**Decisions:** the list holds distinct canonical names sorted alphabetically (as in 2011/2012). The count is its length. It is **empty (unknown)** when `tech_do` was left blank. It is **`0`** when the person answered `tech_do` but ticked no language (only tools such as `Android`): they answered the question without naming a language, which is a different thing from skipping it. There is no `None` option in 2016. Caution for the pooled model: 2016 offered 23 languages (and no `HTML/CSS`, `Bash/Shell`), 2012 offered 13, so a count of 4 does not mean the same in each year; compare within a year or let the model see `survey_year`.

In [35]:
answered = multi_clean["tech_do"].notna()
language_count = pd.Series(np.nan, index=df.index, dtype=float)
language_count[answered] = lang_have[answered].map(lambda s: 0 if pd.isna(s) else len(s.split(";")))
print("rows:", len(df), "| count blank (tech_do unanswered):", int(language_count.isna().sum()), "| count = 0:", int((language_count == 0).sum()))
print("count distribution:")
print(language_count.value_counts(dropna=False).sort_index().to_string())
print("mean count: %.2f" % language_count.mean())
all_names = sorted({n for s in lang_have.dropna() for n in s.split(";")}, key=str.lower)
print("distinct canonical language names:", len(all_names), all_names)
assert len({n.lower() for n in all_names}) == len(all_names), "two spellings of the same name"
for idx in df.index[:5000]:
    s = lang_have[idx]; c = language_count[idx]
    assert (pd.isna(s) and (pd.isna(c) or c == 0)) or c == len(s.split(";")), "count does not match list at " + str(idx)
log("5.3 language count", "LanguageHaveWorkedWith_count = number of distinct languages; blank if tech_do blank; 0 if answered with no language")

rows: 41742 | count blank (tech_do unanswered): 514 | count = 0: 473
count distribution:
0.0       473
1.0      6045
2.0     10552
3.0     11110
4.0      6452
5.0      3294
6.0      1628
7.0       814
8.0       382
9.0       223
10.0      104
11.0       63
12.0       31
13.0       15
14.0        4
15.0        5
16.0        3
17.0        2
18.0        2
19.0        1
20.0        1
21.0        1
22.0        3
23.0       20
NaN       514
mean count: 3.07
distinct canonical language names: 23 ['C', 'C#', 'C++', 'Clojure', 'CoffeeScript', 'Dart', 'F#', 'Go', 'Haskell', 'Java', 'JavaScript', 'MATLAB', 'Objective-C', 'Perl', 'PHP', 'Python', 'R', 'Ruby', 'Rust', 'Scala', 'SQL', 'Swift', 'Visual Basic']
[LOG] 5.3 language count - LanguageHaveWorkedWith_count = number of distinct languages; blank if tech_do blank; 0 if answered with no language


## Gate 6 - Single-answer questions (kept as they are)
### Cell 6.1 - Value sets
Printed so you can see exactly what stays. No merging, no imputation, blanks stay empty. Country, age, gender and the rest keep the file's text. Two things in the file are left alone but worth knowing: `how_to_improve_interview_process` has an option written `interview_coffee` and `why_stack_overflow` has `whyso_onlinepresence` (survey-tool code names, not readable text), and one option reads `Beacause I love to learn` (typo in the survey). I did not invent replacement wording.

In [36]:
SINGLE = [c for c in df.columns if c not in MULTI + ["Unnamed: 0", "salary_range", "salary_midpoint", "experience_range"]]
for c in SINGLE:
    s = df[c]
    print("---", c, "| distinct:", s.nunique(), "| blank:", int(s.isna().sum()))
    if s.nunique() <= 12:
        print(s.value_counts(dropna=False).to_string())

--- collector | distinct: 6 | blank: 0
collector
Meta Stack Overflow Post    21485
Facebook                     7159
House ads                    5550
Reddit                       4974
Twitter                      2499
Miscellaneous                  75
--- country | distinct: 162 | blank: 162
--- un_subregion | distinct: 20 | blank: 314
--- so_region | distinct: 13 | blank: 262
--- age_range | distinct: 9 | blank: 56
age_range
25-29                     12493
20-24                      9259
30-34                      8058
35-39                      4541
40-49                      3874
< 20                       1782
50-59                      1271
> 60                        273
Prefer not to disclose      135
NaN                          56
--- age_midpoint | distinct: 8 | blank: 191
age_midpoint
27.0    12493
22.0     9259
32.0     8058
37.0     4541
44.5     3874
16.0     1782
54.5     1271
65.0      273
NaN       191
--- gender | distinct: 4 | blank: 141
gender
Male                 

## Gate 7 - Headers and assembly
### Cell 7.1 - Header map
**Decisions (review me):** names that appear in your shared list are marked `confirmed` (the 2016 column was matched to them by question meaning, so spot-check `JobSeek`, `RemoteWork`, `EdLevel`): `ResponseId`, `Country`, `Age`, `Gender`, `DevType`, `YearsCode`, `OrgSize`, `Employment`, `EdLevel`, `JobSat`, `JobSeek`, `RemoteWork`, `SOVisitFreq`, `OpSys`, `CompTotal`, `ConvertedCompYearly`, `LanguageHaveWorkedWith`, `LanguageWantToWorkWith`. A few get a `proposed` name (`Industry`, `DevEnvs`, `SOReputation`, `OtherTech...`). I could not see your Canonical Mapping spreadsheet, so check them there. Every other column is a 2016-only question and **keeps its 2016 name** (status `2016 name kept`), so you can map it from the sheet's Raw Column Inventory. `ResponseId` is the file's own first column (the respondent id), not the row number as in 2011/2012. The budget and purchasing questions do not exist in 2016.

In [37]:
RENAME = {  # 2016 column -> (canonical name, status, note)
    "Unnamed: 0": ("ResponseId", "confirmed", "respondent id from the file's unnamed first column"),
    "country": ("Country", "confirmed", "text as in the file"),
    "age_range": ("Age", "confirmed", "text bands, kept as typed"),
    "gender": ("Gender", "confirmed", "kept as typed; 'Prefer not to disclose' and 'Other' are values"),
    "occupation": ("DevType", "confirmed", "single answer in 2016; later years are ';' lists"),
    "experience_range": ("YearsCode", "confirmed", "text bands after removing ' years'; 'Less than 1 year' unchanged"),
    "company_size_range": ("OrgSize", "confirmed", "text bands, kept as typed"),
    "employment_status": ("Employment", "confirmed", "single answer in 2016"),
    "education": ("EdLevel", "confirmed", "';' list in 2016"),
    "job_satisfaction": ("JobSat", "confirmed", "text answers in 2016; later years use a 0-10 score"),
    "open_to_new_job": ("JobSeek", "confirmed", "matched by meaning; check against the sheet"),
    "remote": ("RemoteWork", "confirmed", "matched by meaning; check against the sheet"),
    "visit_frequency": ("SOVisitFreq", "confirmed", "text as in the file"),
    "desktop_os": ("OpSys", "confirmed", "text as in the file"),
    "salary_range": ("CompTotal", "confirmed", "salary BAND text with '<' and '>' labels"),
    "industry": ("Industry", "proposed", ""),
    "dev_environment": ("DevEnvs", "proposed", "';' list; the shared sheet puts coding-environment questions under DevEnvs"),
    "rep_range": ("SOReputation", "proposed", "reputation bands as typed, includes 'I don't have an account' and 'I'm not sure'"),
}
rows = []
for col in EXPECTED_COLS:
    if col == "salary_midpoint":
        rows.append(("(not carried)", "dropped", col, "duplicate of ConvertedCompYearly, equal on every kept row (Cell 3.3)")); continue
    if col == "salary_range":
        rows.append(RENAME[col] + (col,))
        rows.append(("ConvertedCompYearly", "confirmed", "(derived from salary_range)", "annual USD band midpoint; top band midpoint 210,000")); continue
    if col == "tech_do":
        rows += [("LanguageHaveWorkedWith", "confirmed", col, "';' list of canonical languages only; 2016 asks languages and tools in one list"),
                 ("LanguageHaveWorkedWith_count", "proposed", "(derived)", "number of distinct languages; blank if tech_do blank; 0 if answered with no language"),
                 ("OtherTechHaveWorkedWith", "proposed", col, "';' list of the non-language options (Android, AngularJS, SQL Server...)")]; continue
    if col == "tech_want":
        rows += [("LanguageWantToWorkWith", "confirmed", col, "';' list of canonical languages only"),
                 ("OtherTechWantToWorkWith", "proposed", col, "';' list of the non-language options")]; continue
    if col in RENAME:
        n, s, note = RENAME[col]; rows.append((n, s, col, note))
    else:
        rows.append((col, "2016 name kept", col, "2016-only or unmatched; map from the sheet" + ("; ';' list" if col in MULTI else "")))
rows.insert(1, ("survey_year", "proposed", "(added)", "constant 2016"))
header_map = pd.DataFrame(rows, columns=["canonical_name", "status", "original_header", "note"])
print(header_map.status.value_counts().to_dict())
print(header_map[["canonical_name", "status", "original_header"]].to_string(index=False))

{'2016 name kept': 45, 'confirmed': 18, 'proposed': 7, 'dropped': 1}
                  canonical_name         status                          original_header
                      ResponseId      confirmed                               Unnamed: 0
                     survey_year       proposed                                  (added)
                       collector 2016 name kept                                collector
                         Country      confirmed                                  country
                    un_subregion 2016 name kept                             un_subregion
                       so_region 2016 name kept                                so_region
                             Age      confirmed                                age_range
                    age_midpoint 2016 name kept                             age_midpoint
                          Gender      confirmed                                   gender
             self_identification 2016 nam

### Cell 7.2 - Assemble the canonical table
Columns in the file's order with the shared names; `tech_do` and `tech_want` expand into their language / other-technology columns where they were. Blanks stay blank; nothing is imputed, merged or encoded.

In [38]:
out = pd.DataFrame(index=df.index)
for col in EXPECTED_COLS:
    if col == "salary_midpoint":
        continue
    if col == "Unnamed: 0":
        out["ResponseId"] = df[col].astype(int); out["survey_year"] = 2016
    elif col == "salary_range":
        out["CompTotal"] = df[col]; out["ConvertedCompYearly"] = df[col].map(label_to_mid).astype(float)
    elif col == "experience_range":
        out["YearsCode"] = years_code
    elif col == "tech_do":
        out["LanguageHaveWorkedWith"] = lang_have; out["LanguageHaveWorkedWith_count"] = language_count; out["OtherTechHaveWorkedWith"] = other_have
    elif col == "tech_want":
        out["LanguageWantToWorkWith"] = lang_want; out["OtherTechWantToWorkWith"] = other_want
    else:
        name = RENAME[col][0] if col in RENAME else col
        out[name] = multi_clean[col] if col in MULTI else df[col]
assert list(out.columns) == list(header_map.loc[header_map.status != "dropped", "canonical_name"]), "header map and table columns differ"
print("shape:", out.shape)
print(out.head(3).T.to_string(max_colwidth=60))

shape: (41742, 70)
                                                                                            1                                                            3                                                            6
ResponseId                                                                               4637                                                        21378                                                        31743
survey_year                                                                              2016                                                         2016                                                         2016
collector                                                                            Facebook                                                     Facebook                                                     Facebook
Country                                                                           Afghanistan                        

### Cell 7.3 - Validate
Right row count, every row has a salary, no encoded columns, no empty strings, list columns well-formed, single-answer text unchanged from the file, the two edited columns changed only as you asked, missing values left as missing.

In [39]:
assert len(out) == len(df)
assert out["ConvertedCompYearly"].notna().all() or not all(CONFIG.values()), "a kept row has no salary"
assert out["ResponseId"].is_unique
list_cols = ["self_identification", "LanguageHaveWorkedWith", "OtherTechHaveWorkedWith", "LanguageWantToWorkWith", "OtherTechWantToWorkWith", "DevEnvs",
             "EdLevel", "new_job_value", "how_to_improve_interview_process", "star_wars_vs_star_trek", "developer_challenges", "why_stack_overflow"]
for c in list_cols:
    s = out[c].dropna()
    assert not s.str.startswith(";").any() and not s.str.endswith(";").any() and not s.str.contains(";;").any(), c
    assert all(len(v.split(";")) == len(set(v.split(";"))) for v in s), "duplicate item in " + c
    print("%-34s | blank: %5d | mean items: %.2f" % (c, int(out[c].isna().sum()), s.str.split(";").str.len().mean()))
text_cols = [c for c in out.columns if not pd.api.types.is_numeric_dtype(out[c])]
for c in text_cols:
    assert not (out[c].dropna().str.strip() == "").any(), "empty string in " + c
numeric_cols = [c for c in out.columns if c not in text_cols]
print("numeric columns:", numeric_cols)
na_like = {"None", "N/A", "NA", "n/a", "null", "NULL", "NaN", "nan"}
print("cells holding NA-like TEXT that is a real answer:", {c: int(out[c].isin(na_like).sum()) for c in text_cols if int(out[c].isin(na_like).sum())})
untouched = [c for c in SINGLE]
for c in untouched:
    name = RENAME[c][0] if c in RENAME else c
    assert out[name].equals(df[c]), name + " differs from the file"
print("single-answer columns equal the file exactly:", len(untouched), "columns")
assert out["CompTotal"].dropna().map(lambda s: s.startswith("<") or s.startswith(">") or s.startswith("$")).all()
assert not out["CompTotal"].dropna().str.contains("Less than|More than|Greater than", case=False).any()
assert not out["YearsCode"].dropna().str.contains(" years").any()
print("CompTotal has no 'Less than'/'More than'; YearsCode has no ' years'")
print("missing per column (only columns with missing):")
print(out.isna().sum()[out.isna().sum() > 0].to_string())
print("any one-hot style 0/1 columns:", [c for c in numeric_cols if set(out[c].dropna().unique()) <= {0, 1} and c != "survey_year"])

self_identification                | blank:   933 | mean items: 3.04
LanguageHaveWorkedWith             | blank:   987 | mean items: 3.10
OtherTechHaveWorkedWith            | blank: 12915 | mean items: 2.24
LanguageWantToWorkWith             | blank:  3747 | mean items: 2.86
OtherTechWantToWorkWith            | blank: 11256 | mean items: 2.50
DevEnvs                            | blank:  3087 | mean items: 2.57
EdLevel                            | blank:  3680 | mean items: 2.44
new_job_value                      | blank:  4465 | mean items: 5.07
how_to_improve_interview_process   | blank:  6479 | mean items: 2.66
star_wars_vs_star_trek             | blank: 12230 | mean items: 1.11
developer_challenges               | blank:  6532 | mean items: 3.76
why_stack_overflow                 | blank:  6169 | mean items: 3.03
numeric columns: ['ResponseId', 'survey_year', 'ConvertedCompYearly', 'LanguageHaveWorkedWith_count']
cells holding NA-like TEXT that is a real answer: {'hobby': 5481}
sing

### Cell 7.4 - Save and round-trip check

In [40]:
out.to_csv("canonical_2016.csv", index=False)
header_map.to_csv("header_map_2016.csv", index=False)
split_table.to_csv("tech_split_table_2016.csv", index=False)
back = pd.read_csv("canonical_2016.csv", keep_default_na=False, na_values=[""])
assert back.shape == out.shape and list(back.columns) == list(out.columns)
assert (back.isna().values == out.isna().values).all(), "missing-value pattern changed on save"
for c in out.columns:
    if not pd.api.types.is_numeric_dtype(out[c]):
        assert (back[c].fillna("").astype(str).values == out[c].fillna("").values).all(), "text changed on save in " + c
    else:
        assert np.allclose(back[c].fillna(-1).values, out[c].fillna(-1).values), "numbers changed on save in " + c
print("saved canonical_2016.csv | on disk:", back.shape, "| in memory:", out.shape, "| round trip identical")
log("7.4 save", "canonical_2016.csv: " + str(out.shape[0]) + " rows x " + str(out.shape[1]) + " columns, not encoded")
print()
print("row log:")
for step, b0, a0 in row_log:
    print("  %-52s %6d -> %6d" % (step, b0, a0))
print()
print("decision log:")
for step, text in decision_log:
    print(" ", step, "|", text)

C:\Users\KWK\AppData\Local\Temp\ipykernel_25492\3194143241.py:9: Pandas4Warning: 'str' is not supported as a fill value for float64 dtype. In a future version, calling fillna with an incompatible value will raise. Explicitly cast to a common dtype before filling.
  assert (back[c].fillna("").astype(str).values == out[c].fillna("").values).all(), "text changed on save in " + c
C:\Users\KWK\AppData\Local\Temp\ipykernel_25492\3194143241.py:9: Pandas4Warning: 'str' is not supported as a fill value for float64 dtype. In a future version, calling fillna with an incompatible value will raise. Explicitly cast to a common dtype before filling.
  assert (back[c].fillna("").astype(str).values == out[c].fillna("").values).all(), "text changed on save in " + c
C:\Users\KWK\AppData\Local\Temp\ipykernel_25492\3194143241.py:9: Pandas4Warning: 'str' is not supported as a fill value for float64 dtype. In a future version, calling fillna with an incompatible value will raise. Explicitly cast to a common 

saved canonical_2016.csv | on disk: (41742, 70) | in memory: (41742, 70) | round trip identical
[LOG] 7.4 save - canonical_2016.csv: 41742 rows x 70 columns, not encoded

row log:
  1.2 rows loaded                                       56030 ->  56030
  3.2 salary_range blank                                56030 ->  46121
  3.2 salary_range = Rather not say                     46121 ->  43875
  3.2 salary_range = Unemployed                         43875 ->  41995
  3.2 salary_range = Other (please specify)             41995 ->  41742

decision log:
  1.1 encoding | file read with encoding=utf-8 (0 non-ASCII byte(s) in the file)
  1.2 load | loaded 56030 rows x 66 columns, all as text, default NA parsing off
  1.4 strip | 0 column name(s) had surrounding whitespace and were stripped
  1.5 blanks | 700409 empty/whitespace cells converted to NaN; 6277 literal 'None'-style cells kept as text
  3.1 salary labels | 'Less than ' -> '<' and 'More than ' -> '>' applied to salary_range
  3.2 sal